# 03b. 네 가지 흐름 ②: 분기와 루프, 그리고 패턴 번역

03a에서 순차와 병렬을 만들었습니다. 두 흐름은 **업무의 선행 조건만으로** 정해졌습니다. 이 노트북 **03b**는 값을 보고 다음을 정하는 두 흐름, 분기와 루프를 다루고, 마지막에 익숙한 패턴 이름을 네 흐름으로 번역합니다.

## 학습 목표

- 분기와 루프를 `StateGraph`로 직접 만들고 각각의 실패를 재현합니다.
- 라우터의 정확도 하나가 아니라 오분류의 비용으로 분기를 평가합니다.
- 재시도에는 상한과 함께 같은 실패를 알아보는 판정이 있어야 하는 이유를 실행으로 확인합니다.
- Router, Supervisor, Orchestrator-Worker, Handoff, Hierarchical을 네 흐름으로 번역합니다.

## 수업 시간표

| 절 | 내용 | 분 |
|---|---|---:|
| 4 | 분기와 오분류 비용 | 8 |
| 5 | 루프와 종료 조건 | 9 |
| 6 | 패턴을 네 흐름으로 번역 | 7 |
| 7 | 직접 해보기와 정리 | 6 |

## 호출 규모

이 노트북은 **실제 모델**만 씁니다. `Day03_new/.env`의 `ANTHROPIC_API_KEY`가 없으면 첫 코드 셀에서 안내와 함께 멈춥니다. 유료 셀은 실행 직전에 규모를 한 줄로 알려 줍니다. 비용은 본인 키로 청구됩니다.

그래프의 구조, 규칙 라우터, 재시도 시나리오처럼 모델의 판단이 필요 없는 부분은 일반 함수 노드로 돌립니다. 이런 셀은 대부분 출력에 `모델 호출 0회`를 적습니다. 준비 셀, 그래프를 만들거나 그리기만 하는 셀, 앞 실행의 기록을 표로 읽기만 하는 셀은 이 문구가 없어도 모델을 부르지 않습니다.

| 구분 | 유료 셀 수 | 모델 호출(추정) |
|---|---:|---:|
| 모델 라우터 | 1 | 6 |
| 실제 루프 | 1 | 4~6 |
| Orchestrator-Worker 조립 | 1 | 8~15 |
| 기본 실행 합계 | 3 | 약 18~27 |

모델 호출 수는 실행마다 달라집니다. 모델이 도구를 몇 번 쓰는지는 모델이 정하기 때문입니다.

In [ ]:
import json
import operator
import time
from typing import Annotated, Literal, TypedDict

from deepagents import create_deep_agent
from langchain.agents.structured_output import ProviderStrategy
from langchain_core.runnables import RunnableConfig
from langgraph.errors import GraphRecursionError, InvalidUpdateError
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, Send
from pydantic import BaseModel

from mas_lab.contracts import FinalAnswer, Route, WorkerResult, WorkerTask, join_results, merge_results, plan_hash, validate_final
from mas_lab.desk import Desk, load_billing, load_outage
from mas_lab.hybrid import ROLE_GOALS, ROLES
from mas_lab.runner import make_desk
from mas_lab.structures import OUTCOME_GUIDE, gate_answer, scope_line
from mas_lab.trace import ModelMeter, Trace, longest_path, overlap_seconds, timeline
from mas_lab.view import flows_of, show_graph, show_table, write_design_log
from mas_lab.workers import DENY_WRITES, ROLE_TOOLS, allowlist_middleware, contract_prompt, run_worker
from rag_lab.corpus import load_cases
from rag_lab.evidence import BudgetExceeded
from rag_lab.settings import announce, key_status, make_model, require_key

settings = require_key()
model = make_model()
CASES = {case["id"]: case for case in load_cases()}
T1001 = CASES["complex-double-pay"]
desk: Desk = make_desk(T1001, label="점검")
print(key_status(settings))
print("고객 요청 T-1001:", T1001["question"])
print("고객:", desk.scope.entity, "/ 기준일:", desk.scope.as_of, "/ 결제 건수:", len(desk.payments()))
print("측정 장치:", Trace.__name__, ModelMeter.__name__)

## 03a에서 이어 쓰는 값

4~6절은 03a에서 만든 값 몇 개를 다시 씁니다. 이 노트북은 03a의 kernel 없이도 돌아야 하므로, 쓰는 값만 아래 셀에서 다시 정의합니다.

| 값 | 03a에서 | 이 노트북에서 쓰는 곳 |
|---|---|---|
| `CREDIT_PERCENT` | 2절의 순차 그래프 | 4절 Fast Path 실패 재현의 크레딧 계산 |
| `walk` | 2절의 실패 재현 | 4절에서 그래프의 실행 경로 읽기 |
| `PLAN` | 3절의 실제 Worker 실행 | 6절의 Orchestrator-Worker 조립 |
| `gold` | 3절의 필수 사실 검사 | 4절의 라우터 표와 자동 검사, 6절의 자동 검사 |

새로 배우는 내용은 없습니다. 03a를 끝냈다면 읽고 넘어가세요.

In [ ]:
# 03a에서 정의한 값 가운데 이 노트북이 쓰는 것만 다시 정의합니다. 내용은 03a의 해당 셀과 같습니다.

CREDIT_PERCENT = 15  # data/business/sla-a-v2.md의 99.0% 이상 99.5% 미만 구간

def walk(app, initial: dict, **config) -> tuple[list[str], dict]:
    """실행 경로와 마지막 상태를 돌려줍니다. stream의 키가 곧 실행된 Node입니다."""
    path, state = [], dict(initial)
    for update in app.stream(initial, stream_mode="updates", config=config or None):
        for node, change in update.items():
            path.append(node)
            state.update(change or {})
    return path, state

PLAN = {"case_id": T1001["id"], "question": T1001["question"],
        "tasks": [{"task_id": role, "goal": f"{ROLE_GOALS[role]}\n고객 요청: {T1001['question']}", "allowed_tools": sorted(ROLE_TOOLS[role])} for role in ROLES]}

from mas_lab.evaluation import evaluate_row, load_gold, routing_table

gold = load_gold()

print("준비 완료 / 모델 호출 0회")

## 4절. 분기와 오분류 비용

**문제.** 모든 요청에 같은 비용을 쓰지 않으려면 요청을 나눠야 합니다. 나누는 쪽이 틀리면 어떤 일이 생길까요?

Router라는 이름이 거창해 보이지만 실행 구조로는 **조건부 간선**입니다. 앞 절의 `verify` 뒤에 붙인 것과 같습니다. 달라지는 것은 조건을 누가 판단하느냐입니다. 키워드 규칙이 판단할 수도, 모델이 판단할 수도 있습니다. 어느 쪽이든 그래프 입장에서는 함수 하나가 다음 Node 이름을 돌려줄 뿐입니다.

| 경로 | 뜻 | 비용 |
|---|---|---|
| Fast Path | 규정 한 곳이나 결제 한 건만 보면 된다 | 낮음 |
| Slow Path | 결제, 규정, 장애 중 둘 이상이 필요하다 | 높음 |
| 범위 밖 | 현재 고객이 아닌 계약을 묻는다 | 없음. 조사하지 않고 끝낸다 |

**먼저 예측하세요.** 규칙 라우터는 키워드 수를 셉니다. "같은 청구서를 두 번 결제하면 어떻게 처리되나요?"는 단순 질문이지만 규칙은 어느 경로로 보낼까요? 모델 라우터는 이 문장을 어디로 보낼까요? 두 라우터가 어긋날 질문을 하나 더 골라 적어 보세요.

In [ ]:
ROUTE_WORDS = {"결제": ["결제", "환불", "청구"], "장애": ["장애", "가용률", "보상"], "규정": ["규정", "기한", "처리"]}
OTHER_CUSTOMERS = ["베타"]

def needed_kinds(question: str) -> list[str]:
    return [kind for kind, words in ROUTE_WORDS.items() if any(word in question for word in words)]

def rule_route(question: str) -> str:
    if any(name in question for name in OTHER_CUSTOMERS):
        return "out_of_scope"
    return "slow" if len(needed_kinds(question)) >= 2 else "fast"

class RouteState(TypedDict, total=False):
    question: str
    route: str

builder = StateGraph(RouteState)
builder.add_node("classify", lambda state: {"route": rule_route(state["question"])})
for name in ["fast", "slow", "out_of_scope"]:
    builder.add_node(name, lambda state: {})
    builder.add_edge(name, END)
builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", lambda state: state["route"], ["fast", "slow", "out_of_scope"])
rule_app = builder.compile()

show_table([{"조건": "다른 고객 이름이 있다", "다음 Node": "out_of_scope"}, {"조건": "자료 종류가 둘 이상이다", "다음 Node": "slow"}, {"조건": "그 밖", "다음 Node": "fast"}])
samples = ["simple-deadline", "simple-duplicate-rule", "complex-double-pay", "scope-beta-from-alpha"]
show_table([{"사례": c, "질문": CASES[c]["question"], "자료 종류": needed_kinds(CASES[c]["question"]), "경로": " → ".join(walk(rule_app, {"question": CASES[c]["question"]})[0])} for c in samples])
print("모델 호출 0회")

In [ ]:
announce("모델 라우터가 사례 6개를 Route로 분류", model_calls=6)
ROUTER_CASES = ["simple-deadline", "simple-duplicate-rule", "complex-double-pay", "complex-ops-log", "insufficient-no-policy", "scope-beta-from-alpha"]
router_model = model.with_structured_output(Route, method="json_schema")

def model_route(case: dict) -> str:
    prompt = ("고객 요청의 경로를 고르세요. fast: 규정 한 곳이나 결제 한 건만 보면 되는 질문. slow: 결제, 규정, 장애 중 둘 이상이 필요한 질문. "
              f"out_of_scope: 현재 고객이 아닌 다른 고객의 계약을 묻는 질문. {scope_line(make_desk(case))}\n질문: {case['question']}")
    try:
        return router_model.invoke(prompt).path
    except Exception as error:  # 실패도 행으로 남깁니다.
        return f"오류:{type(error).__name__}"

predicted = {case_id: model_route(CASES[case_id]) for case_id in ROUTER_CASES}
table = routing_table(predicted, gold)
show_table([{**row, "규칙 경로": rule_route(CASES[row["사례"]]["question"])} for row in table], ["사례", "기대", "예측", "일치", "규칙 경로"])
print(f"모델 라우터 일치 {sum(row['일치'] for row in table)}/{len(table)} (1회 관찰)")

In [ ]:
# 실패 재현: 복합 요청을 Fast Path로 강제로 보내 수집한 자료를 자동 검사합니다
SLOW_QUERIES = ["중복 결제 환불 규정", "월간 가용률 서비스 크레딧 비율", "서비스 크레딧 신청 기한"]

def collect_fast(case: dict) -> dict:
    """Fast Path: 질문 그대로 문서만 검색합니다."""
    return {"규정": " ".join(e["quote"] for e in make_desk(case).context.search(case["question"], limit=3)["evidence"])}

def collect_slow(case: dict) -> dict:
    """Slow Path: 결제, 가용률, 규정을 모으고 코드로 금액을 계산합니다."""
    desk = make_desk(case)
    payments = [p for p in desk.payments() if p["invoice_id"] == "INV-2026-08"]
    quotes = [e["quote"] for query in SLOW_QUERIES for e in desk.context.search(query, limit=2)["evidence"]]
    return {"결제": json.dumps(payments, ensure_ascii=False), "장애": json.dumps(load_outage(case["entity"]), ensure_ascii=False),
            "규정": " ".join(quotes), "계산": f"크레딧 {payments[0]['base_fee_part'] * CREDIT_PERCENT // 100} 비율 {CREDIT_PERCENT}%"}

def check_collected(label: str, case: dict, collected: dict) -> dict:
    check = evaluate_row({"structure": label, "case_id": case["id"], "answer": " ".join(collected.values())}, gold)
    return {"경로": label, "사례": case["id"], "수집 칸": list(collected), "자료 바이트": len(" ".join(collected.values()).encode()), "누락 사실": check["누락 사실"], "금지 포함": check["금지 포함"]}

show_table([check_collected("Fast 강제", T1001, collect_fast(T1001)), check_collected("Slow", T1001, collect_slow(T1001))])
print("수집한 자료를 검사한 것이지 답을 검사한 것이 아닙니다. 모델 호출 0회")

In [ ]:
class FallbackState(TypedDict, total=False):
    case: dict
    collected: dict
    lacking: list[str]

def fast_node(state):
    return {"collected": collect_fast(state["case"])}

def check_kinds(state):
    return {"lacking": [kind for kind in needed_kinds(state["case"]["question"]) if kind not in state["collected"]]}

def slow_node(state):
    return {"collected": collect_slow(state["case"])}

builder = StateGraph(FallbackState)
builder.add_node("fast", fast_node)
builder.add_node("check_kinds", check_kinds)
builder.add_node("slow", slow_node)
builder.add_edge(START, "fast")
builder.add_edge("fast", "check_kinds")
builder.add_conditional_edges("check_kinds", lambda state: "slow" if state["lacking"] else END, ["slow", END])
builder.add_edge("slow", END)
fallback_app = builder.compile()

rows = []
for case_id in ["simple-deadline", "complex-double-pay"]:
    path, state = walk(fallback_app, {"case": CASES[case_id]})
    rows.append({"경로 기록": " → ".join(path), "부족한 자료 종류": state["lacking"], **check_collected("복구 간선", CASES[case_id], state["collected"])})
show_table(rows, ["사례", "경로 기록", "부족한 자료 종류", "자료 바이트", "누락 사실"])
print("복구 판단은 자료 종류만 봅니다. 평가 파일을 읽지 않습니다. 모델 호출 0회")

### 그림으로 보기: 분기를 어디에 두는가

4절에서 만든 두 그래프를 나란히 놓으면 분기의 자리가 다릅니다. 위는 `rule_app`, 아래는 `fallback_app`입니다.

![분기의 두 자리. 위는 classify가 조건으로 경로를 먼저 고르는 규칙 라우터이고, 아래는 fast를 먼저 돌린 뒤 check_kinds가 자료 종류가 부족하면 slow로 보내는 복구 간선이다](./assets/03b-1-branch-placement.png)

- 위쪽은 일을 시작하기 전에 `classify`가 조건으로 경로를 고릅니다. 경로를 틀리면 그 경로의 비용이나 누락을 그대로 받습니다.
- 아래쪽은 `fast`를 먼저 돌린 뒤 `check_kinds`가 자료 종류를 보고 `slow`로 보낼지 정합니다. 앞으로만 가고, 대신 `fast`에서 쓴 검색이 버려집니다.

**관찰할 것.** 오분류의 비용은 정확도 숫자 하나로 표현되지 않습니다. 방향마다 다른 비용이 듭니다.

- 라우터 표에서 `일치`가 거짓인 행의 방향을 보세요. 기대가 `slow`인데 `fast`로 갔다면 필수 사실이 누락될 위험이고, 기대가 `fast`인데 `slow`로 갔다면 답은 맞아도 호출과 시간을 더 씁니다. 전부 일치했다면 이번 사례 6개와 이번 실행에서는 드러나지 않은 것입니다. 사례가 적고 1회 관찰이라 정확도가 높다고 일반화할 수 없습니다.
- `규칙 경로` 칸과 `기대` 칸을 비교하세요. 규칙은 결과가 항상 같고 비용이 없지만 키워드가 겹치면 그대로 틀립니다. `complex-ops-log`는 두 문서가 필요한 복합 요청인데 키워드로는 자료 종류가 하나로 세어져 Fast Path로 갑니다. 누락 쪽으로 틀리는 방향입니다. 반대로 `simple-duplicate-rule`은 비용 쪽으로 틀립니다. 모델 라우터는 호출이 필요하고 결과가 실행마다 다를 수 있습니다.
- 실패 재현의 `누락 사실` 칸은 복합 요청을 Fast Path로 보냈을 때 무엇을 잃는지 보여 줍니다. 오류는 나지 않습니다. 복합을 단순으로 보내면 누락이 생기고, 단순을 복합으로 보내면 비용이 듭니다.
- 복구 간선 표에서 `simple-deadline`은 `fast`에서 끝나고 `complex-double-pay`는 `slow`까지 갑니다. 복구 판단이 키워드 규칙이므로 키워드가 틀리면 복구도 틀린다는 점도 함께 읽으세요. 복구 간선의 비용은 먼저 쓴 Fast Path의 검색이 버려지는 것입니다.

복구 간선은 앞으로만 갑니다. `slow`가 끝나도 부족하다면 어떻게 할까요? 다시 시도한다면 언제 멈출까요? 다음 절의 질문입니다.

## 5절. 루프와 종료 조건

**문제.** 생성 결과가 검증을 통과하지 못하면 고쳐서 다시 시도합니다. 구조는 단순합니다. 생성 → 검증 → 재시도 또는 종료. 어려운 부분은 언제 멈추느냐입니다.

**먼저 예측하세요.** 검증이 영원히 실패하는 루프에 종료 조건을 넣지 않으면 어떻게 될까요? 끝없이 돌까요, 프레임워크가 멈출까요? 멈춘다면 어떤 이름의 예외로, 어느 단계에서 멈출까요? 한 가지 더. 재시도 상한만 3회로 걸어 두면, 세 번 모두 같은 항목이 빠질 때 이 루프가 한 일은 무엇일까요?

In [ ]:
# 실패 재현: 종료 조건 없는 루프를 recursion_limit로 멈춥니다
class EndlessState(TypedDict, total=False):
    attempts: int

def try_once(state):
    return {"attempts": state["attempts"] + 1}

builder = StateGraph(EndlessState)
builder.add_node("try_once", try_once)
builder.add_edge(START, "try_once")
builder.add_conditional_edges("try_once", lambda state: "try_once", ["try_once", END])
endless_app = builder.compile()

try:
    endless_app.invoke({"attempts": 0}, config={"recursion_limit": 5})
except GraphRecursionError as error:
    print(f"{type(error).__name__}: {str(error).splitlines()[0]}")
print("flows_of:", flows_of(endless_app))
print("모델 호출 0회")

In [ ]:
class RetryState(TypedDict, total=False):
    attempt: int
    history: Annotated[list, operator.add]
    stop: str

def build_retry_loop(find, required: set[str], max_attempts: int, same_limit: int):
    """find(n)은 n번째 시도에서 확보한 항목입니다. same_limit번 연속으로 같은 항목이 빠지면 멈춥니다."""
    def attempt(state):
        number = state.get("attempt", 0) + 1
        return {"attempt": number, "history": [sorted(required - find(number))]}

    def check(state):
        history, last = state["history"], state["history"][-1]
        same = len(history) >= same_limit and all(item == last for item in history[-same_limit:])
        stop = "complete" if not last else "same_failure" if same else "retry_limit" if state["attempt"] >= max_attempts else ""
        return {"stop": stop}

    builder = StateGraph(RetryState)
    builder.add_node("attempt", attempt)
    builder.add_node("check", check)
    builder.add_edge(START, "attempt")
    builder.add_edge("attempt", "check")
    builder.add_conditional_edges("check", lambda state: END if state["stop"] else "attempt", ["attempt", END])
    return builder.compile()

REQUIRED = {"기한", "비율", "기준"}
SCENARIOS = {"나아짐": {1: {"기한"}, 2: {"기한", "비율"}, 3: REQUIRED}, "그대로": {n: {"기한"} for n in range(1, 9)},
             "매번 다름": {n: [{"기한"}, {"비율"}, {"기준"}][n % 3] for n in range(1, 9)}}

def run_scenarios(max_attempts: int, same_limit: int) -> list[dict]:
    rows = []
    for name, found in SCENARIOS.items():
        state = build_retry_loop(lambda n, f=found: f.get(n, set()), REQUIRED, max_attempts, same_limit).invoke({})
        rows.append({"시나리오": name, "종료 상태": state["stop"], "시도": state["attempt"], "시도별 누락": state["history"]})
    return rows

show_table(run_scenarios(max_attempts=5, same_limit=2))
print("모델 호출 0회")

### 그림으로 보기: 루프가 멈추는 길

위 표의 `종료 상태`는 `check`가 아래 순서로 판정한 결과입니다. 위에서부터 차례로 걸리고, 어디에도 걸리지 않으면 `attempt`로 돌아갑니다.

![재시도 루프의 검사 순서. 누락이 없으면 complete, 같은 누락이 반복되면 same_failure, 시도 수가 상한이면 retry_limit으로 멈추고 그 밖에는 attempt로 되돌아간다](./assets/03b-2-loop-exits.png)

- `complete`가 먼저입니다. 누락이 없으면 다른 조건은 보지 않습니다.
- `same_failure`는 `retry_limit`보다 앞입니다. 같은 누락이 반복되면 상한에 닿기 전에 멈춥니다.
- 시도 수와 같은 실패의 연속 횟수는 `max_attempts`와 `same_limit` 두 값이 정합니다. 직접 해보기 1이 이 두 값을 바꿉니다.

위 표의 종료 상태는 세 가지입니다. `complete`는 통과, `same_failure`는 같은 실패의 반복, `retry_limit`은 상한 도달입니다. 세 경우는 다음 행동이 달라야 합니다. `same_failure`는 같은 방법을 더 해도 소용없다는 신호입니다.

이제 같은 구조를 실제 모델로 돌립니다. 초안의 필수 항목을 **코드가** 검사하고, 빠진 항목만 다시 검색해서 다시 씁니다. 모델에게 "빠진 게 있는지 스스로 보라"고 시키지 않습니다. 항목 이름과 값, `evidence_id`를 구조화된 응답으로 받아, 이번 실행에 등록된 근거를 인용했는지 코드로 대조합니다. 값이 없으면 빈칸으로 두도록 지시합니다.

변형으로 코퍼스에 답이 없는 항목(`법인카드 포인트 보상`)을 하나 더해 같은 루프를 다시 돌립니다. 조건을 하나만 바꿔서 멈추는 경로를 봅니다.

In [ ]:
class ItemFill(BaseModel):
    item: str
    value: str = ""
    evidence_id: str = ""

class ItemDraft(BaseModel):
    text: str
    items: list[ItemFill]

class ItemLoopState(TypedDict, total=False):
    question: str
    items: dict
    missing: list[str]
    attempts: Annotated[list, operator.add]
    stop: str

MAX_ATTEMPTS = 3
ITEMS = {"크레딧 비율": "월간 가용률 구간 서비스 크레딧 비율 기본 이용료", "계산 기준": "서비스 크레딧 계산 기준 사용량 초과 요금", "신청 기한": "서비스 크레딧 신청 기한"}
draft_model = model.with_structured_output(ItemDraft, method="json_schema")

def draft_node(state, config: RunnableConfig):
    desk, meter = config["configurable"]["desk"], ModelMeter()
    evidence = [{"evidence_id": e.evidence_id, "quote": e.quote} for e in desk.context.evidence.values()]
    prompt = (f"근거만 써서 답합니다. 필수 항목: {list(state['items'])}. items에 항목 이름을 그대로 쓰고, 원문에 직접 적힌 값과 그 evidence_id를 쓰세요. 원문에 값이 없으면 value와 evidence_id를 비우세요.\n"
              f"{scope_line(desk)}\n질문: {state['question']}\n근거: {json.dumps(evidence, ensure_ascii=False)}")
    reply = draft_model.invoke(prompt, config={"callbacks": [meter]})
    known = desk.known_refs()
    missing = [name for name in state["items"] if not any(f.item == name and f.value and f.evidence_id in known for f in reply.items)]
    totals = meter.totals()
    record = {"시도": len(state.get("attempts", [])) + 1, "근거 수": len(evidence), "누락 항목": missing, "모델 호출": totals["model_calls"], "입력 토큰": totals["input_tokens"], "출력 토큰": totals["output_tokens"], "초안": reply.text}
    return {"missing": missing, "attempts": [record]}

def item_check(state):
    history = [a["누락 항목"] for a in state["attempts"]]
    same = len(history) >= 2 and history[-1] == history[-2]
    return {"stop": "complete" if not history[-1] else "same_failure" if same else "retry_limit" if len(history) >= MAX_ATTEMPTS else ""}

def research_node(state, config: RunnableConfig):
    desk = config["configurable"]["desk"]
    try:
        for name in state["missing"]:  # 빠진 항목만 다시 검색합니다.
            desk.context.search(state["items"][name], limit=2)
    except BudgetExceeded:
        return {"stop": "budget_exceeded"}
    return {}

builder = StateGraph(ItemLoopState)
for name, node in [("draft", draft_node), ("check", item_check), ("research", research_node)]:
    builder.add_node(name, node)
builder.add_edge(START, "draft")
builder.add_edge("draft", "check")
builder.add_conditional_edges("check", lambda state: END if state["stop"] else "research", ["research", END])
builder.add_conditional_edges("research", lambda state: END if state.get("stop") else "draft", ["draft", END])
item_app = builder.compile()
show_graph(item_app)

In [ ]:
announce("초안 → 필수 항목 검증 → 빠진 항목만 재검색 → 재작성, 두 번 실행", model_calls=6)

def run_item_loop(label: str, items: dict) -> dict:
    desk = make_desk(T1001, label=label)
    desk.context.search("서비스 크레딧 비율", limit=1)  # 첫 근거를 일부러 좁게 둡니다.
    state = item_app.invoke({"question": "서비스 크레딧의 비율과 계산 기준, 신청 기한을 알려 주세요.", "items": items}, config={"configurable": {"desk": desk}})
    return {"label": label, "state": state, "desk": desk}

normal_loop = run_item_loop("기본", ITEMS)
no_answer_loop = run_item_loop("답 없는 항목 추가", {**ITEMS, "법인카드 포인트 보상": "법인카드 포인트 보상"})
for loop in (normal_loop, no_answer_loop):
    print(f"[{loop['label']}] 종료 상태 {loop['state']['stop']}, 시도 {len(loop['state']['attempts'])}회, 검색 {loop['desk'].budget.counts.get('search', 0)}회")

In [ ]:
show_table([{"실행": loop["label"], **{k: v for k, v in attempt.items() if k != "초안"}} for loop in (normal_loop, no_answer_loop) for attempt in loop["state"]["attempts"]])
print("[기본] 마지막 초안:", normal_loop["state"]["attempts"][-1]["초안"])
print("[답 없는 항목 추가] 마지막 초안:", no_answer_loop["state"]["attempts"][-1]["초안"])

**관찰할 것.** 시도별 표를 위에서 아래로 읽으세요.

- `누락 항목` 칸이 시도마다 줄어드나요? 줄었다면 재검색이 효과를 낸 것이고, `근거 수` 칸이 늘어난 만큼 다음 호출의 `입력 토큰`도 늘었는지 함께 보세요. 재시도는 공짜가 아닙니다.
- 어떤 시도의 `근거 수`가 앞 시도와 같다면 재검색이 새 근거를 찾지 못한 것이고, 그 시도는 사실상 같은 입력을 다시 보낸 호출입니다. 그렇다면 모델을 부르기 전에 멈추는 규칙을 어디에 넣을 수 있을까요? `research` Node가 무엇을 확인하면 될지 적어 보세요.
- 첫 시도부터 `누락 항목`이 비어 있었다면 모델이 좁은 근거 안에서 값을 채운 것입니다. 이때는 값이 원문에 정말 있는지 초안과 대조하세요. 이 검사는 근거 ID의 소속만 보고 뜻은 보지 않습니다. 그 한계는 다음 노트북에서 다룹니다.
- `답 없는 항목 추가` 실행의 종료 상태를 읽으세요. `same_failure`로 끝났다면 그래프가 같은 실패를 알아채고 멈춘 것이고, 초안이 그 항목을 어떻게 썼는지 읽으세요. 근거가 없다고 밝혔나요? `complete`로 끝났다면 모델이 관계없는 문서로 값을 채운 것일 수 있습니다.
- 두 실행의 첫 시도를 비교하세요. 근거가 똑같이 한 건인데 `누락 항목`이 다르다면 모델이 빈칸을 다르게 판단한 것입니다. 어떤 항목이 `누락 항목`에 없는데 마지막 초안에 구체적인 값이 없다면, 모델이 "약정을 따른다" 같은 문장을 값으로 채웠고 코드가 그것을 통과시킨 것입니다. 빠진 항목만 다시 검색하는 루프는 이런 항목을 다시 찾지 않습니다. 검사가 약한 만큼 루프도 일찍 끝납니다.
- 앞의 결정적 표와 비교하세요. `same_failure`와 `retry_limit`은 멈춘 이유가 다릅니다. 같은 프롬프트로 다시 부르는 것은 전략이 아닙니다. 재시도에서 달라지는 것은 프롬프트에 든 **근거**여야 합니다.

이제 네 흐름을 다 만들었습니다. 다음 절에서 패턴 이름을 번역합니다.

## 6절. 패턴을 네 흐름으로 번역

**문제.** 새로운 패턴 이름은 계속 나옵니다. 이름마다 새로 배우면 끝이 없고, 이름이 다르다는 이유로 다른 구조라고 믿으면 비교를 못 합니다. 이 절은 익숙한 다섯 이름을 네 흐름으로 번역합니다. 번역이 되면 이름 없이 "순서는? 독립은? 조건은? 종료는?"으로 비교할 수 있습니다.

| 패턴 | 순차 | 병렬 | 분기 | 루프 | 한 줄 번역 |
|---|---|---|---|---|---|
| Router | 분류 뒤 처리 | 없음 | 있음 | 없음 | 조건부 간선 하나 |
| Supervisor | 지시 뒤 보고 | 필요하면 | 있음 | 있음 | 분기 + 루프 |
| Orchestrator-Worker | 분해 → 집계 | 있음 | 검증 뒤 | 재계획 | 분해 → 병렬 → 집계 |
| Handoff | 담당자 전환 | 없음 | 있음 | 왕복 | 담당자 전환 + 반복 |
| Hierarchical | 층 사이 | 층 안에서 | 층마다 | 층마다 | 위 구조의 중첩 |

**먼저 예측하세요.** 아래에서 일반 함수로 다섯 뼈대를 만들고 `flows_of`로 분석합니다. `flows_of`는 그래프의 간선을 세어 조건부 간선 수와 순환 유무를 냅니다. Supervisor는 Worker를 차례로 부르고, Hierarchical은 팀 안에서 `Send`로 두 명을 동시에 돌립니다. 두 뼈대의 `flows_of` 칸은 얼마나 다를까요? 그 칸만 보고 어느 쪽이 병렬인지 구별할 수 있을까요?

In [ ]:
class SkeletonState(TypedDict, total=False):
    log: Annotated[list, operator.add]
    turns: int

def step(name: str, seconds: float = 0.05):
    """구간을 기록하고 잠깐 기다리는 일반 함수 노드입니다."""
    def node(state, config: RunnableConfig):
        with config["configurable"]["trace"].span(name):
            time.sleep(seconds)
        return {"log": [name], "turns": state.get("turns", 0) + 1}
    return node

def member(payload: dict, config: RunnableConfig):
    with config["configurable"]["trace"].span(payload["task_id"]):
        time.sleep(0.05)
    return {"log": [payload["task_id"]]}

def graph_of(nodes: dict, edges: list, conditional: list):
    builder = StateGraph(SkeletonState)
    for name, node in nodes.items():
        builder.add_node(name, node)
    for start, end in edges:
        builder.add_edge(start, end)
    for start, chooser, targets in conditional:
        builder.add_conditional_edges(start, chooser, targets)
    return builder.compile()

fan_two = lambda state: [Send("member", {"task_id": f"member_{n}"}) for n in ("a", "b")]
router_skeleton = graph_of({"route": step("route"), "fast": step("fast"), "slow": step("slow")}, [(START, "route"), ("fast", END), ("slow", END)],
                           [("route", lambda s: "slow", ["fast", "slow"])])
supervisor_skeleton = graph_of({"supervisor": step("supervisor"), "worker_a": step("worker_a"), "worker_b": step("worker_b")},
                               [(START, "supervisor"), ("worker_a", "supervisor"), ("worker_b", "supervisor")],
                               [("supervisor", lambda s: {1: "worker_a", 3: "worker_b"}.get(s["turns"], END), ["worker_a", "worker_b", END])])
orchestrator_skeleton = graph_of({"plan": step("plan"), "member": member, "aggregate": step("aggregate")},
                                 [(START, "plan"), ("member", "aggregate"), ("aggregate", END)], [("plan", fan_two, ["member"])])
print("뼈대 3개:", [flows_of(app)["노드"] for app in (router_skeleton, supervisor_skeleton, orchestrator_skeleton)], "개 노드")

In [ ]:
def hand_over(name: str, other: str, state: dict, config: RunnableConfig) -> Command:
    with config["configurable"]["trace"].span(name):
        time.sleep(0.05)
    turns = state.get("turns", 0) + 1
    return Command(update={"log": [name], "turns": turns}, goto=END if turns >= 3 else other)

def owner_a(state, config: RunnableConfig) -> Command[Literal["owner_b", "__end__"]]:
    return hand_over("owner_a", "owner_b", state, config)

def owner_b(state, config: RunnableConfig) -> Command[Literal["owner_a", "__end__"]]:
    return hand_over("owner_b", "owner_a", state, config)

def team(name: str):
    inner = graph_of({"lead": step(f"{name}_lead"), "member": member}, [(START, "lead"), ("member", END)], [("lead", fan_two, ["member"])])
    def node(state, config: RunnableConfig):  # 부모의 log를 다시 합치지 않도록 새 입력으로 부릅니다.
        return {"log": inner.invoke({"log": []}, config=config)["log"], "turns": state.get("turns", 0) + 1}
    return node

handoff_skeleton = graph_of({"owner_a": owner_a, "owner_b": owner_b}, [(START, "owner_a")], [])
hierarchical_skeleton = graph_of({"boss": step("boss"), "team_a": team("team_a"), "team_b": team("team_b")},
                                 [(START, "boss"), ("team_a", "boss"), ("team_b", "boss")],
                                 [("boss", lambda s: {1: "team_a", 3: "team_b"}.get(s["turns"], END), ["team_a", "team_b", END])])
SKELETONS = {"Router": router_skeleton, "Supervisor": supervisor_skeleton, "Orchestrator-Worker": orchestrator_skeleton,
             "Handoff": handoff_skeleton, "Hierarchical": hierarchical_skeleton}
flow_rows = [{"패턴": name, **flows_of(app)} for name, app in SKELETONS.items()]
show_table(flow_rows)
print("모델 호출 0회")

In [ ]:
rows = []
for flow_row in flow_rows:
    trace = Trace(flow_row["패턴"])
    state = SKELETONS[flow_row["패턴"]].invoke({"log": []}, config={"configurable": {"trace": trace}})
    rows.append({**flow_row, "겹침(초)": overlap_seconds(trace.spans), "병렬": overlap_seconds(trace.spans) > 0, "실행 순서": " → ".join(state["log"])})
show_table(rows, ["패턴", "조건부 간선", "순환", "겹침(초)", "병렬", "실행 순서"])
print("모델 호출 0회")

`조건부 간선`과 `순환` 칸은 그래프를 분석해서 얻고, `겹침(초)`와 `병렬` 칸은 **실행해서** 얻었습니다. Supervisor와 Hierarchical 행을 나란히 읽어 보세요. 앞의 칸은 같은데 `병렬` 칸만 다릅니다. Hierarchical의 팀 내부 병렬은 부모 그래프의 간선에 보이지 않기 때문입니다. Orchestrator-Worker처럼 `Send`가 부모 그래프에 있어도 조건부 간선 하나로 그려질 뿐이어서, 하나를 고르는 분기인지 여럿을 보내는 병렬인지는 간선만으로 구별되지 않습니다. 정적 분석과 실행 trace는 서로를 대신하지 못합니다.

이제 하나를 끝까지 조립합니다. Orchestrator-Worker를 `plan → Send → aggregate → validate → replan`으로 이어 네 흐름이 어디 있는지 Node마다 표시합니다. 먼저 일반 함수로 돌립니다. `outage`가 첫 시도에서 실패하도록 해 두었습니다. 재계획은 실패한 작업만 `outage#2`처럼 새 키로 다시 보냅니다. `merge_results`는 같은 키의 다른 값을 거절하므로, 재시도 결과를 같은 키에 덮어쓸 수 없습니다. 그래서 새 키를 쓰고, `join_results`가 같은 작업의 마지막 시도를 읽습니다.

### 그림으로 보기: Orchestrator-Worker에서 네 흐름이 있는 자리

위에서 설명한 조립이 아래 셀의 `build_orchestrator`가 만드는 그래프입니다. 노트북의 `FLOW_OF_NODE`가 Node마다 적은 흐름을 이름표로 붙였습니다.

![Orchestrator-Worker 조립 그래프. plan, worker, aggregate는 순차이고 worker는 Send로 병렬이며 validate는 분기이고 replan에서 worker로 되돌아가는 간선이 루프다](./assets/03b-3-orchestrator-flows.png)

- `plan → worker → aggregate`는 순차이고, `worker`는 `Send`로 여러 개가 같은 단계에서 돕니다.
- `validate`의 조건부 간선이 분기이고, `replan`에서 `worker`로 되돌아가는 간선이 루프입니다.
- `replan` 뒤에 다시 보낼 작업이 없으면 `aggregate`로 바로 갑니다(점선). 이때는 Worker 결과를 두고 답만 다시 씁니다.

In [ ]:
class OrchestratorState(TypedDict, total=False):
    tasks: list[dict]
    results: Annotated[dict, merge_results]
    final: dict
    problems: list[str]
    replans: int
    problem_log: Annotated[list, operator.add]
    order: Annotated[list, operator.add]

FLOW_OF_NODE = {"plan": "순차: 분해가 먼저입니다", "worker": "병렬: Send로 같은 단계에 실행합니다", "aggregate": "순차: 모두 끝나야 합칩니다",
                "validate": "분기: 통과, 재계획, 포기", "replan": "루프: 부족한 작업만 worker로 돌아갑니다"}

def build_orchestrator(tasks: list[dict], work, write, check, max_replans: int = 1):
    def send_tasks(state):
        return [Send("worker", {"task": t}) for t in state["tasks"]] or "aggregate"
    def worker(payload, config: RunnableConfig):
        return {"results": {payload["task"]["task_id"]: work(payload["task"], config)}, "order": [f"worker:{payload['task']['task_id']}"]}
    def replan(state, config: RunnableConfig):
        number = state.get("replans", 0) + 1
        info = join_results({k: WorkerResult.model_validate(v) for k, v in state["results"].items()}, [t["task_id"] for t in tasks])
        retry = [{**t, "task_id": f"{t['task_id']}#{number + 1}", "attempt": number + 1} for t in tasks if t["task_id"] in info["failed"] + info["missing"]]
        return {"tasks": retry, "replans": number, "order": ["replan"]}
    def validate(state, config: RunnableConfig):
        problems = check(state, config)
        return {"problems": problems, "problem_log": [problems], "order": ["validate"]}
    nodes = {"plan": lambda state: {"tasks": tasks, "order": ["plan"]}, "worker": worker, "replan": replan, "validate": validate,
             "aggregate": lambda state, config: {"final": write(state, config), "order": ["aggregate"]}}
    builder = StateGraph(OrchestratorState)
    for name, node in nodes.items():
        builder.add_node(name, node)
    builder.add_edge(START, "plan")
    builder.add_conditional_edges("plan", send_tasks, ["worker", "aggregate"])
    builder.add_edge("worker", "aggregate")
    builder.add_edge("aggregate", "validate")
    builder.add_conditional_edges("validate", lambda s: "replan" if s["problems"] and s.get("replans", 0) < max_replans else END, ["replan", END])
    builder.add_conditional_edges("replan", send_tasks, ["worker", "aggregate"])
    return builder.compile()

def worker_problems(state: dict, required: list[str]) -> list[str]:
    info = join_results({k: WorkerResult.model_validate(v) for k, v in state["results"].items()}, required)
    return [f"{name}: 결과 없음" for name in info["missing"]] + [f"{name}: 실패" for name in info["failed"]]

TASK_IDS = [t["task_id"] for t in PLAN["tasks"]]
show_table([{"Node": node, "흐름": flow} for node, flow in FLOW_OF_NODE.items()])
print("입력은 앞의 PLAN 그대로입니다:", TASK_IDS)

In [ ]:
def plain_work(task: dict, config: RunnableConfig) -> dict:
    with config["configurable"]["trace"].span(task["task_id"]):
        time.sleep(0.05)
    if task["task_id"] == "outage":
        return WorkerResult(task_id="outage", status="failed", error="주입한 첫 시도 실패").model_dump()
    return WorkerResult(task_id=task["task_id"], attempt=task.get("attempt", 1), status="complete", facts=[f"{task['task_id']} 확인"], evidence_refs=["F-예시"]).model_dump()

plain_app = build_orchestrator(PLAN["tasks"], plain_work, lambda state, config: {"작업 수": len(state["results"])},
                               lambda state, config: worker_problems(state, TASK_IDS))
trace = Trace("조립")
plain_state = plain_app.invoke({}, config={"configurable": {"trace": trace}})
print("실행 순서:", " → ".join(plain_state["order"]))
print("결과 키:", sorted(plain_state["results"]), "/ 재계획:", plain_state.get("replans", 0))
print("검증별 문제:", plain_state["problem_log"])
print(timeline(trace.spans))
show_graph(plain_app)
print("flows_of:", flows_of(plain_app), "/ 모델 호출 0회")

In [ ]:
def live_work(task: dict, config: RunnableConfig) -> dict:
    desk = config["configurable"]["desk"]
    try:
        return run_worker(WorkerTask.model_validate(task), desk, model).model_dump()
    except BudgetExceeded as error:
        return WorkerResult(task_id=task["task_id"], status="failed", error=str(error)).model_dump()

def live_write(state: dict, config: RunnableConfig) -> dict:
    desk = config["configurable"]["desk"]
    facts = {k: {f: v.get(f) for f in ("status", "facts", "evidence_refs", "uncertainty", "error")} for k, v in state["results"].items()}
    allowed = ROLE_TOOLS["writer"]
    agent = create_deep_agent(model=model, tools=desk.tools("writer"), system_prompt=f"고객에게 보낼 답을 씁니다. {scope_line(desk)}\n{OUTCOME_GUIDE}",
                              middleware=allowlist_middleware(desk.budget, allowed), permissions=DENY_WRITES, response_format=ProviderStrategy(FinalAnswer), name="writer")
    content = (f"고객 요청: {T1001['question']}\nWorker 결과: {facts}\n앞선 검증 문제: {state.get('problems') or '없음'}\n"
               "크레딧 금액은 calculate_credit으로 계산하고, 기본 이용료분만 계산 기준에 넣으세요. 실패하거나 누락된 작업은 uncertainty에 쓰세요.")
    with desk.trace.span("writer", "write"):
        reply = agent.invoke({"messages": [{"role": "user", "content": content}]}, config={"recursion_limit": 40})
    return reply["structured_response"].model_dump()

def live_check(state: dict, config: RunnableConfig) -> list[str]:
    desk = config["configurable"]["desk"]
    return worker_problems(state, TASK_IDS) + validate_final(FinalAnswer.model_validate(state["final"]), desk.known_refs(), desk.source_text())

live_app = build_orchestrator(PLAN["tasks"], live_work, live_write, live_check)
print("앞 셀과 같은 build_orchestrator입니다. 바꾼 것은 work, write, check 세 함수입니다.")
print("flows_of:", flows_of(live_app))

In [ ]:
announce("Orchestrator-Worker 조립을 실제 Worker로 T-1001에 실행", model_calls=15)
live_desk, live_meter, started = make_desk(T1001, label="조립"), ModelMeter(), time.perf_counter()
live_state = live_app.invoke({}, config={"callbacks": [live_meter], "configurable": {"desk": live_desk}})
live_seconds = round(time.perf_counter() - started, 3)
live_final = gate_answer(FinalAnswer.model_validate(live_state["final"]), live_desk, approved=None, request_id="flows-t1001")
cited = sorted({e.doc_id for ref, e in live_desk.context.evidence.items() if ref in live_final.evidence_refs})
live_check_row = evaluate_row({"structure": "조립", "case_id": T1001["id"], "answer": live_final.text, "uncertainty": live_final.uncertainty, "outcome": live_final.outcome,
                               "cited_docs": cited, "ledger_refunds": len(live_desk.ledger.refunds), "status": "ok"}, gold)
totals = live_meter.totals()
print("실행 순서:", " → ".join(live_state["order"]))
print("검증별 문제:", live_state["problem_log"])
print(timeline(live_desk.trace.spans))
show_table([{"전체(초)": live_seconds, "겹침(초)": overlap_seconds(live_desk.trace.spans), "모델 호출": totals["model_calls"], "입력 토큰": totals["input_tokens"], "재계획": live_state.get("replans", 0),
             "남은 문제": live_state["problems"], "outcome": live_final.outcome, "원장 건수": len(live_desk.ledger.refunds), "자동 검사": live_check_row["자동 검사"], "실패 칸": live_check_row["실패 칸"], "누락 사실": live_check_row["누락 사실"]}])
print(live_final.text)

**관찰할 것.** 네 흐름으로 쓰면 패턴 이름 없이도 비교할 수 있습니다. 위 출력을 흐름별로 읽으세요.

- 순차: `실행 순서`에서 `plan`이 맨 앞이고 `aggregate`가 Worker 뒤에 있습니다. `writer` 구간은 timeline에서 Worker 구간이 다 끝난 뒤에 시작합니다.
- 병렬: `겹침(초)`가 0보다 큰지 봅니다. 이 값은 코드가 정한 병렬의 결과이므로 모델의 선택과 관계가 없습니다. 다만 Worker 구간의 길이는 모델이 도구를 쓴 횟수에 따라 달라집니다.
- 분기와 루프: `재계획` 칸이 0이면 첫 `validate`에서 끝난 것입니다. 1이면 `검증별 문제`의 첫 항목이 재계획을 부른 원인입니다. `실행 순서`에서 `replan` 바로 뒤에 `worker:`가 있으면 실패한 Worker를 다시 돌린 것이고, `aggregate`가 바로 오면 Worker 결과는 두고 답만 다시 쓴 것입니다. 앞의 일반 함수 실행과 어느 쪽이 같은지 보세요. 마지막 `남은 문제`가 비어 있지 않다면 상한에 걸려 문제를 남긴 채 끝난 것이며, 이 문제를 고객 답과 함께 드러내는 일은 다음 노트북의 주제입니다.
- `outcome`이 `needs_approval`이고 `원장 건수`가 0이면 게이트가 멈춘 것입니다. 쓰기는 이 노트북에서 일어나지 않습니다. `자동 검사`가 거짓이면 `실패 칸`과 `누락 사실`을 읽고, 표현 차이인지 정말 빠졌는지 위 답과 대조하세요. 문자열 포함 검사는 문맥을 모릅니다.

새로운 패턴 이름을 만나면 순서는 무엇인지, 무엇이 독립인지, 어떤 조건으로 갈라지는지, 언제 끝나는지부터 물으세요. 네 질문에 답이 있으면 그 패턴은 이미 만들 수 있고, 다른 구조와 비교할 수 있습니다. 답이 없다면 그 이름은 아직 실행 구조를 말하지 않은 것입니다.

## 직접 해보기

수강생이 자기 업무로 바꿔 보는 칸입니다. 두 칸 모두 모델을 부르지 않습니다. 고친 뒤 그대로 실행하면 됩니다.

### 해보기 1

재시도 상한과 같은 실패 판정을 바꿔 종료 상태를 비교하세요. `same_limit`을 크게 하면 `그대로` 시나리오는 어떻게 끝나나요? 상한을 2로 줄이면 `나아짐`은 끝까지 가지 못하나요? 업무마다 어떤 값이 맞을지 적어 보세요.

In [ ]:
MY_MAX_ATTEMPTS = 5
MY_SAME_LIMIT = 4
show_table(run_scenarios(max_attempts=MY_MAX_ATTEMPTS, same_limit=MY_SAME_LIMIT))
print("모델 호출 0회")

### 해보기 2

03a의 해보기 1에서 쓴 업무(`MY_TASKS`)나 자기 업무 하나를 골라, 네 흐름이 어디에 있는지 적으세요. 순차와 병렬은 선행 조건에서, 분기와 루프는 "값에 따라 달라지는 곳"과 "다시 하는 곳"에서 찾습니다. 이 칸은 설계 기록이라 정답이 없습니다. 06이 이 기록을 모아 보여 주므로 자기 말로 쓰세요.

In [ ]:
MY_FLOWS = """순차: (내 업무에서 앞 작업이 끝나야 시작하는 것)
병렬: (서로 결과가 필요 없는 작업과, 동시에 돌리면 얻는 시간)
분기: (다음 작업을 가르는 조건과, 잘못 갈랐을 때의 비용)
루프: (다시 시도하는 조건과 멈추는 조건)"""
print(write_design_log("작업 Graph: 순차·병렬·분기·루프는?", MY_FLOWS).name, "에 기록했습니다.")
print(MY_FLOWS)

## 정리

| 흐름 | 그래프 도구 | 막는 실패 | 재현한 실패 |
|---|---|---|---|
| 분기 | 조건부 간선 | 모든 요청에 같은 비용 | 복합 요청의 Fast Path 누락 |
| 루프 | 순환 간선, 상한, 같은 실패 판정 | 고칠 수 있는 실패의 포기 | `GraphRecursionError` |

패턴은 이렇게 번역됩니다. Router는 분기입니다. Supervisor는 분기와 루프이고 필요하면 병렬을 씁니다. Orchestrator-Worker는 분해 → 병렬 → 집계이고 검증 뒤 재계획이 루프입니다. Handoff는 담당자 전환과 반복입니다. Hierarchical은 그 중첩입니다.

`flows_of`는 간선을 세지만 실제로 겹쳤는지는 실행 trace만 압니다. 순차와 병렬의 시간 효과를 어떻게 비교하는지는 03a의 정리에 있습니다.

## 확인 질문

1. 라우터의 정확도 하나로 비용을 판단하면 무엇을 놓치나요?
2. Supervisor를 이 노트북의 네 흐름으로 번역하세요. `flows_of`가 말해 주지 못하는 칸은 무엇인가요?
3. 재시도 상한만 있고 같은 실패 판정이 없는 루프는 어떤 비용을 쓰나요? 재시도에서 무엇이 달라져야 하나요?

다음 노트북은 이 네 흐름을 운영할 수 있게 만듭니다. 오늘 Worker 사이에 넘긴 결과를 계약으로 정하고, 예산을 걸고, 실패 종류마다 다음 행동을 나누고, 늦게 도착한 결과와 승인, 멱등을 그래프에 넣습니다.